# Introduction 

* Over the past four weeks we explored various data preprocessing techniques and solved some regression problems using linear and logistic regression models. The other side of the supervised learning paradigm is classification probelms.
* To solve such problems we are going to consider image classification as a running example and solving it using `Perceptron()` method.

# Imports

What is the first step?
* Ya, import all necessary packages. For classification problems, we need to import classes and utilities from `sklearn.linear_model.`
 * This module has implementations for different classification models like `Perceptron, LogisticRegression, svm` and `knn`.

 We also need to import a bunch of model selection utilities from `sklearn.model_selection` module and metrics from `sklearn.metrics` module.

The data preprocessing utilities are imported from `sklearn.preprocessing` modules.


In [1]:
#Common imports 
import numpy as np
import os
import io 
import warnings

#sklearn specific imports
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import Perceptron
from sklearn.metrics import hinge_loss
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_curve
from sklearn.metrics import precision_score, recall_score, classification_report
from sklearn.model_selection import cross_validate, cross_val_predict, GridSearchCV 
from pprint import pprint
#To plot pretty figures.


# Handwritten Digit Classification

* We are going to use **Perceptron Classifier** to classify (recognize) given digit images. Since a single perceptron could only be used for binary classification, We consider only two classes in the first half. Eventually we extend it to multi-class setting.
* Suppose we want to recognize whether the given image is of digit zero or not(digit other than zero). Then the problem could be cast as a binary classification problem.
* The first step is to create a dataset that contains a collection of digit images(also called examples, samples) written by humans. Then each image should be labelled properly. Daunting task! 
* Fortunately, we have a standard benchmark dataset called **MNIST**. well, why not make use of itt? Let us import the dataset first...

# Data loading and splitting.

In [ ]:
X,y = fetch_openml('mnist_784',version=1, return_X_y=True)
#It returns Data and Label as a pandas dataframe

The data matrix $X$ and the respective label vector $ y$ need to be converted to the numpy array by calling a `to_numpy` method.


In [ ]:
X=X.to_numpy()
y=y.to_numpy()


* Let's get some information like number of features, number of classes about the dataset.
* Observe that the labels are of string data type not integers.


In [ ]:
target_names = np.unique(y)
print("Number of samples:{0}, type:{1}".format(X.shape[0],X.dtype))
print("Number of features:{0}".format(X.shape[1]))
print("Minimum:{0},Maximum:{1}".format(np.min(X),np.max(X)))
print("Number of classes:{0},type:{1}".format(len(target_names),y.dtype))
print("Labels: {0}".format(target_names))


* The **MNIST** dataset is clean and the range of values that each feature can take is also known. Therefore, the samples in the dataset may not require many data preprocessing techniques.
* However, it is often better to scale the range of features between 0 to 1.
* So, we can either use `MinMaxScaler` or `MaxAbsScaler`. They don't make any difference as the image pixels can takes only positive value from 0 to 255.


In [ ]:
X = MinMaxScaler().fit_transform(X)
print("Minimum:{0}, Maximum{1}".format(np.min(X),np.max(X)))

#Data Visualization

Let us pick a few images(the images are already shuffled in the dataset) and display them with their respective labels. As said above, the images are stacked as a row vector of size $ 1 \times 784$ and therefore must be reshaped to the matrix of size $ 28 \times 28$ to display them properly.


In [ ]:
import matplotlib.pyplot as plt
num_images = 9 # Choose a square number 
factor = np.int(np.sqrt(num_images))
fig,ax = plt.subplots(nrows=factor, ncols=factor,figsize=(8,6))
idx_offset = 0 # take "num_images" starting from the index "idx_offset"
for i in range(factor):
  index = idx_offset+ i*(factor)
  for j in range(factor):
    ax[i,j].imshow(X[index+j].reshape(28,28), cmap='gray')
    ax[i,j].set_title('Label:{0}'.format(str(y[index+j])))
    ax[i,j].set_axis_off()


If you closely observe, you can see that there are moderate variations in the appearance of digits (say, digit: 1) These matrices also close to sparse (that is , there are lots of 0(black pixels) in the matrix than non -zero pixels)



#Data splitting.
* Now, we know the details such as number of samples, size of each sample, number of features(784), number of classes (targets) about the dataset.
* So let us split the total number of samples into train and test set in the following ratio: 60000/10000 (that is, 60000 samples in the training set and 10,000 samples in the testing set).
* Since the samples in the data set are already randomly shuffled, we need **not to** shuffle it again. Therefore using `train_test_split()` may be skipped.


#Binary Classification : 0-Detector

In [ ]:
x_train, x_test, y_train, y_test = X[:60000],X[60000:],y[:60000],y[60000:]


Before procedding further, we need to check whether the dataset is balanced or imbalanced. We can do it by plotting the distribution of samples in each classes.


In [ ]:
import seaborn as sns
plt.figure(figsize=(10,4))
sns.histplot(data=np.int8(y_train),binwidth=0.45, bins=11)
plt.xticks(ticks=[0,1,2,3,4,5,6,7,8,9],labels=[0,1,2,3,4,5,6,7,8,9])
plt.xlabel('Class')
plt.title('Distribution of Samples')
plt.show()

#Modifying Labels

* Let us start with a simple classification problem, that is binary classification.
* Since the original label vector contains **10** classes, we need to modify the number of classes to 2.
* Therefore, the label **0** will be changed **1** and all other labels(1-9) will be changed to **-1**.
* We name the label vectors as `y_train_0` and `y_test_0`.


In [ ]:
#initialize new variable names with all -1
y_train_0 = -1*np.ones(len(y_train))
y_test_0 = -1*np.ones(len(y_test))

#find indices of digit 0 image
indx_0 = np.where(y_train=='0') #remember original labels are of type str not int use those indices to modify y_train_0&y_test_0
y_train_0[indx_0]=1
indx_0=np.where(y_test=='0')
y_test_0[indx_0]=1

#Sanity check: ✅
* Let's display the elements of y_train and y_train_0 to verify whether the labels are properly modified. Of course, we can't verify all the 60000 labels by inspection (unless we have a plenty of time or man power 😃)


In [ ]:
print(y_train)
import matplotlib.pyplot as plt
num_images = 9 # Choose a square number 
factor = np.int(np.sqrt(num_images))
fig,ax = plt.subplots(nrows=factor, ncols=factor,figsize=(8,6))
idx_offset = 0 # take "num_images" starting from the index "idx_offset"
for i in range(factor):
  index = idx_offset+ i*(factor)
  for j in range(factor):
    ax[i,j].imshow(X[index+j].reshape(28,28), cmap='gray')
    ax[i,j].set_title('Label:{0}'.format(str(y_train_0[index+j])))
    ax[i,j].set_axis_off()


#Baseline Models

Enough about Data! 

Let us quickly construct a baseline model with the following rule(you are free to choose different rule)

1. Count number of samples per class.
2. The model **always outputs** the class which has highest number of samples.
3. Then calculate the accuracy of the baseline model.



In [ ]:
num_pos = len(np.where(y_train_0==1)[0])
num_neg = len(np.where(y_train_0==-1)[0])
print(num_pos,num_neg)


In [ ]:
base_clf = DummyClassifier(strategy='most_frequent') # there are other strategies

In [ ]:
base_clf.fit(x_train,y_train_0)
print("Training accuracy:{0:.2f}".format(base_clf.score(x_train, y_train_0)))
print("Testing accuracy:{0:.2f}".format(base_clf.score(x_test,y_test_0)))

* Now the reason is obvious. The model would have predicted 54077 sample correctly just by outputing -1 for all the input samples. Therefore the accuracy will be $ \frac{54077}{60000}=90.12 \% $

This is the reason why "accuracy" alone is not always a good measure!.


#Perceptron model 
$ //  //$ 
Before using perceptron for binary classification, it will be helpful to recall the important concepts (equations) covered in technique course.

#Recap(Theory) 


Let us quickly recap various components in the general settings:

1. **Training data**(features label or $(\mathbf X,y)$ where $y$ is a **discrete** number from a finite set **Features** in this case are **pixel**  values of an image.
2. **Model**: 
\begin{eqnarray} h_w:y&=&\text g(\mathbf w^T 
\mathbf x) \\ 
&=&\text g(w_0+w_1x_1+\ldots + w_mx_m)\end{eqnarray} where,
 * $\mathbf w$ is weight vector in $\mathbb{R}^{(m+1)}$ i.e. it has components: $\{w_0,w_1,\ldots,w_m\}$
 * g$(z)$ is a non-linear activation function given by a signum function:

$$\text g(z)=\begin{cases} +1 ,\text {if} \ z \ge 0 \\
-1, \text {otherwise}(i.e. z \lt 0)\end{cases}$$

3. **Loss function**: Let $ {\hat y}^{(i)} \in \{-1,+1\}$ be the prediction from perceptron and ${\hat y}^{(i)}$ be the actual label for $i-\text{th}$ example. 
$ \\ $

The error is 

$$\text e^{(i)}=\begin{cases} 0 , \ \ \text {if} \ \ {\hat y}^{(i)} = y^{(i)} \\
-\mathbf {w^Tx^{(i)}}y^{(i)}, \text {otherwise}(i.e. {\hat y}^{(i)} \ne y^{(i)})\end{cases}$$

THis can be compactly written as:
\begin{equation} e^{(i)}=\max(-\mathbf{w^Tx^{(i)}}y^{(i)},0)=\max(-h_{\text w }(\mathbf x^{(i)})y^{(i)},0)\end{equation} 

4.**Optimization**:
 * Perceptron learning algorithm
 1. Initialize $\mathbf {\text w}^{(0)}=0$
 2. For each training example $(x^{(i)},y^{(i)})$
  * ${\hat y}^{(i)}=\text{sign}(\mathbf {w^Tx}^{(i)})[\text {calculate the output value}]$
  * $\mathbf w^{(t+1)} := \mathbf w^{(t)}+ \alpha (y^{(i)}-{\hat y}^{(i)})\mathbf x^{(i)}[\text{Update the weights}] $

   Linearly separable examples lead to convergence of the algorithm with zero training loss, else it oscillates.

#Prameters of Perceptron Class

* Let's quickly take a look into the important parameters of the Perceptron()
`class sklearn.linear_model.Perceptron(*,penalty=None, alpha = 0.0001, l1_ration=0.15, fit_intercept = True, max_iter=1000,tol=0.001, shuffle=True, verbose=0, eta0=1.0, n_jobs=None, random_state=0, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5,class_weight=None, warm_start=False).`
* Need not to pay attention to all the arguments and their default values.
* Internally, the API uses the preceptron loss (i.e. it calls **Hinge(0,0)**, where 0.0 is a threshold) and uses SGD to update the weights.
* You may refer to the documentation for more details on the `Perceptron` class.
* The other way of deploying perceptron is to use the general `linear_model.SGDClassifier` with `loss='perceptron'`

* The above loss is termed as hard Hinge-loss (as scores pass through the sign function) and hence we can't use SGD.
* Whereas, SKlearn implements hinge-Loss with the following definition: $\max(0,-wx^iy^i$) and by default calls sgd to minimize the loss.

#Instantiation
* Create an instantiation of binary classifier (bin_clf) and call `fit` method to train the model.



In [ ]:
bin_clf = Perceptron(max_iter=100,random_state=1729)


#Training and Prediction

* Call the `fit` method to train the model
* It would be nice to plot the iteration vs loss curve for the training. However, sklearn does not have a direct function to plot it.
* Nevertheless, we can workaround this using `partial_fit` method (Which will be demonstrated at the end of the lecture)


In [ ]:
bin_clf.fit(x_train, y_train_0)
print("Dimension of Weights:{0}".format(bin_clf.coef_.shape))
print("Bias:{0}".format(bin_clf.intercept_))
print("The loss function:{0}".format(bin_clf.loss_function_))


Let us make predictions on the train set and then calculate the training accuracy.

In [ ]:
y_hat_train_0 = bin_clf.predict(x_train)
print("Training Accuracy:", bin_clf.score(x_train,y_train_0))


Let us make the predictions on the test set and then calculate the testing accuracy.


In [ ]:
print('Test accuracy:',bin_clf.score(x_test,y_test_0))


#Displaying predictions
* Take few images from the testset at random and display it with the corresponding predictions.
* Plot a few images in a single figure window along with their respective **Predictions**. 


In [ ]:
y_hat_test_0 = bin_clf.predict(x_test)
num_images = 9 # choose a square number 
factor = np.int(np.sqrt(num_images)) 
fig,ax = plt.subplots(nrows=factor, ncols = factor, figsize=(8,6))
idx_offset  = 0 # display "num_images" starting from idx_offset
for i in range(factor):
  index = idx_offset + i*(factor)
  for j in range(factor):
    ax[i,j].imshow(x_test[index+j].reshape(28,28),cmap='gray')# we should not use x_train_with_
    ax[i,j].set_title("Prediction:{0}".format(str(y_hat_test_0[index+j])))
    ax[i,j].set_axis_off()

In [ ]:
indx_0 = np.where(y_test_0==1)

In [ ]:
indx_0  = np.where(y_test_0==1)
zeroImgs = x_test[indx_0[0]]
zeroLabls = y_hat_test_0[indx_0[0]]
num_images = 9 #Choose a square number 
factor = np.int(np.sqrt(num_images))
fig, ax = plt.subplots(nrows=factor, ncols=factor, figsize=(8,6))
idx_offset = 0 # display "num_images" starting from idx_offset 
for i in range(factor):
  index = idx_offset + i*(factor)
  for j in range(factor):
    ax[i,j].imshow(zeroImgs[index+j].reshape(28,28),cmap='gray') # we should not use x_train_with_
    ax[i,j].set_title("prediction:{0}".format(str(zeroLabls[index+j])))
    ax[i,j].set_axis_off()

It seems that there are a significant number of images that are correctly classified. Let's see how many?


In [ ]:
num_misclassified = np.count_nonzero(zeroLabls==-1)
num_correctpred= len(zeroLabls)-num_misclassified
accuracy = num_correctpred/len(zeroLabls)
print(accuracy)

* This above score (guess the name of the metric) is less than the accuracy score of the model but it seems preety descent!. 
* Will it be the same if we consider another digit, say, 5 for positive class and all other class as negative?..Of course not. You may cross check it.**(Take it as an excercise)**

#Better Evaluation metrics
* We now know that using the accuracy **alone** to measure the performance of the model is not suitable(especially for imbalanced datasets), so which are the more suitable metrics then?

#**Confusion Matrix**



In [ ]:
y_hat_train_0 = bin_clf.predict(x_train)
cm_display = ConfusionMatrixDisplay.from_predictions(y_train_0,y_hat_train_0, values_format='.5g',display_labels=bin_clf.classes_)
plt.show()

* Pay attention to the number of FPs and FNs. Suppose for some reasons, we want the classifer to avoid FPs to a good extent irrespective of FNs, how can we achive it?
* To answer it, let's compute the other metrics which take FPs and FNs into account.
#Precision and Recall

* We can use the function `classification_report` to compute these parameters. However, for the time being let's compute these parameters using the data from the confusion matrix manually (Not a difficult thing to do, right 🧑?)

In [ ]:
cf_matrix = cm_display.confusion_matrix 
tn = cf_matrix[0,0]
fn = cf_matrix[1,0]
fp = cf_matrix[0,1]
tp = cf_matrix[1,1]

In [ ]:
precision = tp/(tp+fp) 
print("Precision:",precision)
recall = tp/(tp+fn)
print('Recall:',recall)
accuracy = (tn+tp)/(tn+tp+fn+fp)
print('accuracy:',accuracy)


* The precision is close to 0.98. Despite it, we still want to increase the precision. Let's come back to this later.
* In general, we would like to know whether the model under consideration with the set hyper-parameters is a good one for a given problem.

#Cross validation

* Well to address this, we have to use cross-validation folds and measure the same metrics across these folds for different values of hyper-parameters.
* However, perceptron does not many hyperparameters other than the learning rate.
* For the moment, we set the learning rate to its default value. Later, we use `GridSearchCV` to find the better value for the learning rate.



In [ ]:
bin_clf = Perceptron(max_iter=100,random_state=1729) #repeating for readability
scores = cross_validate(bin_clf, x_train,y_train_0,cv=5, scoring=['precision','recall','f1'],return_estimator=True)
pprint(scores)


* **NOTE** 

The perceptron estimator passed as an argument to the function `cross_validate` is internally cloned `num_fold (cv=5)` times and fitted independently on each fold. (you can check this by setting `warm_start=True`)

* Compute the average and standard deviation of scores for all three metrics on (k=5) folds to measure the generalization!.


In [ ]:
print('f1,             avg:{0:.2f},  std:{1:.3f}'.format(scores['test_f1'].mean(), scores['test_f1'].std()))
print('precision,      avg:{0:.2f},  std:{1:.2f}'.format(scores['test_precision'].mean(), scores['test_precision'].std()))
print('recall,         avg:{0:.2f},  std:{1:.2f}'.format(scores['test_recall'].mean(), scores['test_recall'].std()))

* Let us pick the first estimator returned by the cross-validate function.
* So, we can hope that the model might also perform well on test data. Let's check that out..


In [ ]:
bin_clf = scores['estimator'][0]
y_hat_test_0 = bin_clf.predict(x_test)
cm_display = ConfusionMatrixDisplay.from_predictions(y_test_0, y_hat_test_0, values_format='.5g')

In [ ]:
print('Precision:{0:.2f}'.format(precision_score(y_test_0,y_hat_test_0)))
print('recall:{0:.2f}'.format(recall_score(y_test_0,y_hat_test_0)))


This is good ! 

**WAY-2 for Generalization:**
(Optional)
* There is an **another approach** of getting predicted labels via cross-validation and using it to measure the generalization. 
* In this case, each sample in the dataset will be part of only one test set in the splitted folds.

In [ ]:
y_hat_train_0 = cross_val_predict(bin_clf, x_train, y_train_0,cv=5)


In [ ]:
cm_display = ConfusionMatrixDisplay.from_predictions(y_train_0,y_hat_train_0, values_format='.5g')
plt.show()

In [ ]:
cf_matrix = cm_display.confusion_matrix 
tn = cf_matrix[0,0]
fn = cf_matrix[1,0]
fp = cf_matrix[0,1]
tp = cf_matrix[1,1]


In [ ]:
precision = tp/(tp+fp) 
print("Precision:",precision)
recall = tp/(tp+fn)
print('Recall:',recall)
accuracy = (tn+tp)/(tn+tp+fn+fp)
print('accuracy:',accuracy)


* Compare the precision and recall score obtained by the above method with that of the previous method(i.e. using `cross_validate`) 
* Finally, we can print all these scores as a report using the `classification_report` function

In [ ]:
print('Precision:{0:.2f}'.format(precision_score(y_train_0,y_hat_train_0))) 
print('Recall:{0:.2f}'.format(recall_score(y_train_0,y_hat_train_0)))
print('-'*53)
print(classification_report(y_train_0,y_hat_train_0))

#Precision/Recall Tradeoff 

* Often time we need to make a trade off between precision and recall scores of a model.
* It depends on the problem at hand.
* It is important to note that we should **not** pass the **predicted labels** as input to `precision_recall_curve` function, instead we need to pass the probability scores or the output from the decision function!.
* The `Perceptron()` class contains a `decision_function` method, therefore we can make use of it.
* Then, internally the decision scores are sorted, **tps** and **fps** will be computed by changing the threshold from index[0] to index [-1].
* Let us compute the scores from decision function.


In [ ]:
bin_clf = Perceptron(random_state=1729)
bin_clf.fit(x_train,y_train_0)
y_scores = bin_clf.decision_function(x_train)

sns.histplot(np.sort(y_scores))
plt.show()

Can you think why there are so many negative values than the positives ?
**Hint** Class-Imbalance
* Suppose threshold takes the value of -600, then all the samples having score greater than -600 is set to 1(Positive label) and less than it is set to -1(neg label) 
* Therefore, the number of False Positives will be increased. This will in turn reduce the precision score to a greater extent.
* On the otherhand, if the threshold takes the value of say 400, Then, the number of False negatives will be increase and hence the recall will reduce to a greater extent.

* Let's see it in action.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_train_0,y_scores,pos_label=1)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(precisions[:-1],recalls[:-1],"b--")
plt.xlabel('Precision')
plt.ylabel('Recall')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(thresholds,precisions[:-1],"b--",label="Precision")
plt.plot(thresholds,recalls[:-1],"g-",label="Recall")
plt.xlabel('Threshold')
plt.grid(True)
plt.legend(loc='upper right')
plt.show()

In [ ]:
#get the index of threshold around zero 
idx_th = np.where(np.logical_and(thresholds >0, thresholds <1))
print("precision for zero threshold:",precisions[idx_th[0][0]])

* **Here is the solution** to the question how can we increase the precision of the classifier by compromising the recall. We can make use of the above plot.
* Let's see how.

#The ROC Curve


In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
fpr, tpr, thresholds =roc_curve(y_train_0, y_scores)
plt.figure(figsize=(10,4))
plt.plot(fpr, tpr, linewidth =2, label='Perceptron')
plt.plot([0,1],[0,1],'k--', label='baseEstimator')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True)
plt.legend()
plt.show() 


#Warm start vs Cold Start 
$ \\ \\ $ 
#Cold Start 

* If we execute the `fit` method of `bin_clf` repeatedly, we get the same score for both training and testing accuracy.
* This because everytime the `fit` method is called, the model weights are initialized to the same values. Therefore, we obtain the same score.
* This is termed as **cold start** Let's execute the following cell 4 times tand observe the score.

In [ ]:
bin_clf.fit(x_train,y_train_0)
y_hat_train_0 = bin_clf.predict(x_train)
print("Training Accuracy:", bin_clf.score(x_train, y_train_0))
print("Test accuracy", bin_clf.score(x_test, y_test_0))

#Warm Start

* As you might have guessed, there is an approach called `warm Start`.
* Setting `warm_start=True` retains the weight values of the model after `max_iter` and hence produce different results for each execution.
* Warm starting is useful in many ways. It helps us train the model by initializing the weight values from the previous state. So, we can pause the training and resume it whenever we get the resource for computation.
* Of course, it is not required for simple models like perceptron and for a small dataset like **MNIST**.
* In this notebook, we use this feature to plot the iteratation vs loss curve.
* Let us execute the following lines of code 4 times and observe how the training accuracy changes for each execution.

In [ ]:
bin_clf_warm = Perceptron(max_iter=100,random_state=1729,warm_start=True)

In [ ]:
bin_clf_warm.fit(x_train,y_train_0)
print("Training Accuracy:", bin_clf_warm.score(x_train,y_train_0))

#Multiclass Classifier (OneVsAll)
* We know that the perceptron is a binary classifier. However,MNIST dataset contains 10 classes. Then how can we extend the idea to handle multi-class problem?

* **Solution**: Combine multiple binary classifiers and devise a suitable scoring metric.
* Sklearn makes it extremely easy without modifying a single line of code that we have written for the binary classifier.

* Sklearn does this by counting a number of unique elements (10 in this case) in the label vector `y_train` and converting labels using `Labelbinarizer` to fit each binary classifier (Remember binary classifier requires binary labels, Tautology 😃) 
* That's all!


In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import LabelBinarizer

In [ ]:
clf = Perceptron(random_state=1729)


In [ ]:
# let's use label binarizer just to see the encoding
y_train_ovr = LabelBinarizer().fit_transform(y_train)

In [ ]:
for i in range(10):
  print("{0}:{1}".format(y_train[i],y_train_ovr[i]))

* The `y_train_ovr` will be of size of size $60000 \times 10$.
* The first column will be (binary) label vector for 0-detector 😃and the next one for 1-Detector and so on.


In [ ]:
clf.fit(x_train,y_train)

* What had actually happened internally was that the API automatically created 10 binary classifiers, converted labels to binary sparse matrix and trained them with the binarized labels !
* During the inference time, the input will be passed through all these 10 classifiers and the highest score among the output from the classifiers will be considered as the predicted class.
* To see it in action, let us execute the following lines of code.


In [ ]:
print ('Shape of Weight matrix: {0} and bias vector:{1}'.format(clf.coef_.shape, clf.intercept_.shape))

* So it is a matrix of size $ 10 \times 784 $ Where each row represents the weights for a single binary classifier.
* Important difference to note is that there is no signum function associated with the perceptron.
* The class of a perceptron that outputs the maximum score for the input sample is considered as the predicted class.


In [ ]:
for i in range(10):
  scores = clf.decision_function(x_train[i].reshape(1,-1))
  #print(scores)
  #print("The predicted class:", np.argmax(scores))
  #print()
  print("Predicted output:")
  print(clf.predict(x_train[i].reshape(1,-1)))
  


In [ ]:
# get the prediction for all training samples
y_hat = clf.predict(x_train)

In [ ]:
print(classification_report(y_train,y_hat))

Let us display the confusion matrix and relate it with the report above.

In [ ]:
cm_display = ConfusionMatrixDisplay.from_predictions(y_train,y_hat, values_format='.5g') #it return matplotlib plot object 



* What are all the insights we could infer from the above figure?
* Digit 2 is often confused with Digit 3 (Reasonable!) 
# Making a Pipeline

* Let's create a pipeline to keep the code compact.
* Recall that, the MNIST dataset is clean and hence doesn't require much preprocessing.
* The one potential preprocessing technique we may use is to scale the features within the range(0,1)
* It is **not** similar to scaling down the range values between 0 and 1.


In [ ]:
# create a list with named tuples
estimators = [('std_scaler',MinMaxScaler()),('bin_clf',Perceptron())]
pipe = Pipeline(estimators)

In [ ]:
pipe.fit(x_train,y_train_0)

In [ ]:
y_hat_train_0=pipe.predict(x_train)
cm_display = ConfusionMatrixDisplay.from_predictions(y_train_0,y_hat_train_0,values_format='.5g')
plt.show()

#Iteration vs Loss Curve 
The other way of **Plotting Iteration Vs Loss Curve** with the `Partial_fit` method.


In [ ]:
iterations =100 
bin_clf1 = Perceptron(max_iter=100,random_state=2094)
Loss_clf1=[]
for i in range(iterations):
  bin_clf1.partial_fit(x_train,y_train_0,classes=np.array([1,-1])) 
  y_hat_0 = bin_clf1.decision_function(x_train)
  Loss_clf1.append(hinge_loss(y_train_0,y_hat_0))


In [ ]:
plt.figure()
plt.plot(np.arange(iterations),Loss_clf1)
plt.grid(True)
plt.xlabel('Iteration')
plt.ylabel('Training Loss')
plt.show()

#GridSearchCV 
* So, far we didn't do any hyperparameter tuning. We accepted the default value for learning rate of the Perceptron class.
* Now, let us search for a better learning rate using `GridSearchCV`.
* No matter what the learning rate is, the loss will never converge to zero as the classes are not linearly separable.

In [ ]:
from sklearn.metrics import make_scorer
scoring = make_scorer(hinge_loss,greater_is_better=False)
lr_grid = [1/2**n for n in range(1,6)]
bin_clf_gscv = GridSearchCV(Perceptron(), param_grid={'eta0':lr_grid},scoring=scoring, cv=5)
bin_clf_gscv.fit(x_train,y_train_0) 


In [ ]:
pprint(bin_clf_gscv.cv_results_)

As you can see, the best learning rate is 0.125 


In [ ]:
iterations =100
Loss = []
best_bin_clf = Perceptron(max_iter=1000,random_state=2094,eta0=0.125)
for i in range(iterations):
  best_bin_clf.partial_fit(x_train, y_train_0, classes=np.array([1,-1]))
  y_hat_0 = best_bin_clf.decision_function(x_train)
  Loss.append(hinge_loss(y_train_0,y_hat_0))

In [ ]:
plt.figure()
plt.plot(np.arange(iterations),Loss_clf1,label='eta0=1')
plt.plot(np.arange(iterations),Loss, label='eta0=0.125')
plt.grid(True)
plt.legend()
plt.xlabel("Iteration")
plt.ylabel("Training Loss")
plt.show()

Well, instead of instantiating a Perceptron class with a new learning rate and re-train the model, we could simply get the best_estimator from `GridSearchCV` as follows.


In [ ]:
best_bin_clf = bin_clf_gscv.best_estimator_ 

In [ ]:
y_hat_train_0 = bin_clf.predict(x_train) 
print(classification_report(y_train_0, y_hat_train_0))

Compare the classification report when `eta0=1`

#Visualizing weight vectors (Optional)

It will be interesting to look into the samples which are misclassified as False Positives (that is, images that are not zero but classified as zero), and come up with some possible reasons. Shall we do it?


In [ ]:
#repeating the code for readability 
bin_clf = Perceptron(max_iter=100)
bin_clf.fit(x_train,y_train_0)
y_hat_train_0 = bin_clf.predict(x_train)

In [ ]:
#find the index of false positive samples
idx_n = np.where(y_train_0==-1)#index of true -ve samples
idx_pred_p = np.where(y_hat_train_0==1) #index of predicted positive samples
idx_pred_n = np.where(y_hat_train_0==-1) #index of predicted negative samples
idx_fp = np.intersect1d(idx_n, idx_pred_p) 
idx_tn = np.intersect1d(idx_n,idx_pred_p)

In [ ]:
fig, ax = plt.subplots(nrows=factor, ncols=factor, figsize=(8,6))
idx_offset = 0 
for i in range(3):
  index = idx_offset + i
  for j in range(3):
    ax[i,j].imshow(x_train[idx_fp[index+j]].reshape(28,28),cmap='gray') #we should not use x_train_with_dummy 
    ax[i,j].set_title('GT:{0},Pr:{1}'.format(str(y_train_0[idx_fp[index+j]]),str(y_hat_train_0[idx_fp[index+j]])))
    ax[i,j].set_axis_off() #GT ground truth

In [ ]:
from matplotlib.colors import Normalize

In [ ]:
w = bin_clf.coef_ 
w_matrix = w.reshape(28,28)
#fig = plt.figure() 
#plt.imshow(w_matrix, cmap='magma')
#plt.imshow(w_matrix, cmap='cividis')
#plt.imshow(w_matrix, cmap='viridis')
#plt.imshow(w_matrix, cmap='gray')
plt.imshow(w_matrix, cmap='inferno')
#plt.grid(False)
#plt.axis(False)
plt.colorbar()
plt.show()

In [ ]:
#print(idx_fp.shape)
activation = w * x_train[idx_fp[0]].reshape(1,-1) 
lin_out = activation.reshape(28,28) 
plt.subplot(1,2,1) 
plt.imshow(x_train[idx_fp[0]].reshape(28,28),cmap='inferno')
plt.colorbar() 
#lin_out[lin_out < 0]=0 # just set the value less than zero to zero 
plt.subplot(1,2,2)
plt.imshow(lin_out,cmap='plasma')
plt.colorbar() 
plt.grid(False) 
plt.axis(False) 
plt.show() 


In [ ]:
# input to the signum
print(np.sum(lin_out)+bin_clf.intercept_)

In [ ]:
activation = w*(x_train[idx_tn[0]].reshape(1,-1)) 
lin_out = activation.reshape(28,28) 
plt.subplot(1,2,1)
plt.imshow(x_train[idx_tn[0]].reshape(28,28),cmap='plasma') 
plt.colorbar()
lin_out[lin_out<0]=0 #just set the value less than zero to zero 
plt.subplot(1,2,2) 
plt.imshow(lin_out,cmap='plasma') 
plt.colorbar() 
plt.grid(False) 
plt.axis(False) 
plt.show() 


In [ ]:
# input to signum 
print(np.sum(lin_out) + bin_clf.intercept_)



---



---



---



---
# **Week 6** Naive Bayes classifier  

#Imports 

In this notebook we solve the same problem of recognizing Handwritten digits using Logistic regression model.




In [ ]:
#Common imports 
import numpy as np
from pprint import pprint  

#to make this notebook's output stable across runs
np.random.seed(42) 

#sklearn specific imports 
from sklearn.preprocessing import MinMaxScaler 
from sklearn.pipeline import make_pipeline 
from sklearn.dummy import DummyClassifier 
from sklearn.linear_model  import SGDClassifier, RidgeClassifier, LogisticRegression 
from sklearn.model_selection import cross_validate, RandomizedSearchCV, cross_val_predict 
from sklearn.metrics import log_loss # log loss is also known as cross entropy loss 
from sklearn.metrics import ConfusionMatrixDisplay 
from sklearn.metrics import precision_score, recall_score, classification_report 
from sklearn.metrics import precision_recall_curve 
from sklearn.metrics import roc_curve, roc_auc_score 

#scipy 
from scipy.stats import loguniform 
# To plot pretty figures 
%matplotlib inline 
import matplotlib as mpl 
import matplotlib.pyplot as plt 
import seaborn as sns 

#global settings 
mpl.rc('axes',labelsize=14) 
mpl.rc('xtick',labelsize=12) 
mpl.rc('ytick',labelsize=12) 
mpl.rc('figure',figsize=(8,6))


In [ ]:
#Ignore all warnings (convergence..) by sklearn 
def warn(*args,**kwargs): 
  pass
import warnings  
warnings.warn = warn


#Handwritten Digit Classification 
* We are going to use LogisticRegression (Despite it's name) to classify (recognize) given digit image. Again, we first apply the model for binary classification and then extend it to multiclass classification.

* Suppose we want to recognize whether the given image is of digit zero or not (digits other than zero). Then the problem could be case as binary classification problem.
 *The first step is to create a dataset that contains collection of digit images (also called examples, samples) written by humans. Then each image should be labelled properly. Daunting taks! 
 * Fortunately, we have a standard benchmark dataset called **MNIST**. Well, why not make use of it? Let us import the dataset first...
 

In [ ]:
from sklearn.datasets import fetch_openml 
X_pd,y_pd = fetch_openml('mnist_784',version=1,return_X_y=True) #it returns Data and label as a panda dataframe.


 

The data matrix $X$ and the respective label vector $y$ need to be converted to numpy array by calling a `to_numpy` method.


In [ ]:
#import numpy as np
X = X_pd.to_numpy() 
y = y_pd.to_numpy() 


#Pre-Processing 
* Unlike perceptron, where scaling the range is optional(but recommended), sigmoid requires range between 0 to 1.
* Contemplate the consequence if we don't apply the scaling operation on the input datapoints.
* Note: **Do not** apply mean centering as it removes zeros from the data, however zeros should be zeros in the dataset.
* Sine we are using only one preprocessing step, using `pipeline` may not be required.



In [ ]:
scaler = MinMaxScaler() 
X = scaler.fit_transform(X) 


In [ ]:
print("Mean of the features:", np.mean(X))
print("Standard Deviation:", np.std(X))
print("Minimum value", np.min(X))
print("Maximum value:", np.max(X))

Let's get some information about the dataset. (Note that the labels are of string data type) 

#Data splitting

In [ ]:
x_train, x_test, y_train, y_test = X[:60000],X[60000:],y[:60000],y[60000:]


#Binary Classification : 0-Detector 
* Let us start with a simple classification problem, that is, binary classification.
* Since the original label vector contains 10 classes, we need to modify the number of classes to 2. Therefore, the label '0' will be changed to '1' and all other labels(1-9) will be changed to '0' 
* **(Note: for perceptron we set the negative labels to -1)**


In [ ]:
# initialize new variables names with all 0.
y_train_0 = np.zeros((len(y_train)))
y_test_0 = np.zeros((len(y_test)))

#find indices of digit 0 image
indx_0 = np.where(y_train=='0') #remember original labels are of type str not int 
# use those indices to modify y_train_0 & y_test_0 
y_train_0[indx_0] = 1 
indx_0 = np.where(y_test == '0')
y_test_0[indx_0]=1



In [ ]:
import matplotlib.pyplot as plt
num_images = 9 # Choose a square number 
factor = np.int(np.sqrt(num_images))
fig,ax = plt.subplots(nrows=factor, ncols=factor,figsize=(8,6))
idx_offset = 0 # take "num_images" starting from the index "idx_offset"
for i in range(factor):
  index = idx_offset+ i*(factor)
  for j in range(factor):
    ax[i,j].imshow(X[index+j].reshape(28,28), cmap='plasma')
    ax[i,j].set_title('Label:{0}'.format(str(y_train_0[index+j])))
    ax[i,j].set_axis_off()

#Baseline Models
Let us quickly construct a baselinen model with the following rule 
1. Count number of samples per class.
2. The model **always output** the class which has highest number of samples.
3. Then calculate the accuracy of the baseline model.


In [ ]:
num_pos = len(np.where(y_train_0==1)[0])
num_neg = len(np.where(y_train_0==0)[0])
print(num_pos, num_neg)

In [ ]:
base_clf = DummyClassifier(strategy='most_frequent') #there are other approches too
base_clf.fit(x_train,y_train_0)
print(base_clf.score(x_train,y_train_0))

Now the reason is obvious. The model would have predicted 54077 sample correctly just by outputing 0 for all the input samples. Therefore the accuracy will be $\frac{54077}{60000}=90.12 \%$

#LogisticRegression model 
Before using LogisticRegression for Binary classification problem, it will be helpful to recall the important concepts (equations) covered in the technique course.

##Recap

Let us quickly recap various  components in the general settings:
  1. **Training data**: (features,label) or $(\mathbf X,y)$, where label $y$ is a **discrete** number from a finite set. **Features** in this case are pixel values of an image.
  2. Model: 
$$ z = w_0x_0 + w_1x_1+ \ldots + w_mx_m$$
$$ = \mathbf {w}^T \mathbf x$$ and passing it through the sigmoid non-linear function (or Logistic function)
$$ \sigma(z)=\frac{1}{1+e^{-z}}$$
3. Loss function: 
\begin{equation} J(\mathbf w) = -\frac{1}{n} \mathbf \sum [y^{(i)} \log(h_w(\mathbf x^{(i)}))+(1-y^{(i)})(1-\log(h_w(\mathbf x^{(i)})))] \end{equation}

4. **Optimization**
Gradient Descent 

* Let's look into the parameters of the `SGDClassifier()` estimator:
`class sklearn.linear_model.SGDClassifier(loss='hinge', *,penalty='l2', alpha=0.0001, l1_ratio = 0.15, fit_intercept =True, max_iter =1000, tol=0.001, shuffle=True, verbose =0, epsilon=0.1, n_jobs=None, random_state=None, learning_rate = 'optimal', eta0=0.0, power_t = 0.5, early_stopping = False, validation_fraction =0.1, n_iter_no_change=5, class_weight=None, warm_start=False, average=False)`.

* Setting the loss parameter to `loss=log` makes it a logistic regression classifier. We may refer to documentation for more details on the `SGDClassifier` class.
* Create an instant of binary classifier(bin_sgd_clf) and call the `fit` method to train the model.
*Let us use fit method of `SGDClassifier()` to plot the iteration vs loss curve(Of course, we could use `partial_fit()` method as well) 
* Therefore, to capture the loss for each iterations during training we set the parameters `warm_start =True` and `max_iter=1`

#Training without regularization

* Set `eta0=0.01,learning_rate='constant' ` and `alpha=0`.

In [ ]:
bin_sgd_clf =SGDClassifier(loss='log',
                           penalty='l2',
                           warm_start=True,
                           eta0=0.01,
                           alpha=0,
                           learning_rate='constant',
                           random_state=1729)
Loss=[] 
iterations=100
for i in range(iterations):
  bin_sgd_clf.fit(x_train,y_train_0)
  y_pred=bin_sgd_clf.predict_proba(x_train)
  Loss.append(log_loss(y_train_0,y_pred))


In [ ]:
plt.figure() 
plt.plot(np.arange(iterations),Loss)
plt.grid(True)
plt.xlabel('Iterations')
plt.ylabel('Label') 
plt.show()

Let us calculate the training and testing accuracy of the model.


In [ ]:
print('Training accuracy:{0:.2f}'.format(bin_sgd_clf.score(x_train,y_train_0)))
print('Testing accuracy{0:.2f}'.format(bin_sgd_clf.score(x_test,y_test_0)))

* We know that accuracy alone is not a good metric for binary classification.
* Let's compute Precision,recall and f1-score for the model.


In [ ]:
y_hat_train_0 = bin_sgd_clf.predict(x_train)
cm_display = ConfusionMatrixDisplay.from_predictions(y_train_0,y_hat_train_0,values_format='.5g')
plt.show()

In [ ]:
print(classification_report(y_train_0,y_hat_train_0))

DO Cross validation to check for the generalization ability of the model.


In [ ]:
estimator = SGDClassifier(loss='log',
                          penalty='l2',
                          max_iter=100,
                          warm_start=False,
                          eta0=0.01,
                          alpha=0,
                          learning_rate='constant',
                          random_state=1729)

In [ ]:
cv_bin_clf = cross_validate(estimator,x_train,y_train_0,cv=5,
                            scoring=['precision','recall','f1'],
                            return_train_score=True,
                            return_estimator=True)
pprint(cv_bin_clf)

* From the above result, we can see that logistic regression is better than the perceptron.! 

* However, it is good to check the weight values of all the features and decide whether regularization could be of any help.


In [ ]:
weights = bin_sgd_clf.coef_
bias = bin_sgd_clf.intercept_
print('Bias:',bias)
print(weights.shape,bias.shape)
plt.figure() 
plt.imshow(weights.reshape(28,28),cmap='inferno')
plt.grid(False)
plt.colorbar()
plt.show()

In [ ]:
plt.figure() 
plt.plot(np.arange(0,784),weights[0,:])
plt.ylim(np.min(weights[0])-5,np.max(weights[0])+5)
plt.grid(True)
plt.xlabel('Feature Index')
plt.ylabel('Weight value')
plt.show()


* It is interesting to observe how many weight values are exactly zero.
* Those features contribute nothing in the classification.


In [ ]:
zero_weight_idx = np.where(weights[0]==0)
print(len(zero_weight_idx[0]))

#num_zero_w = weights.shape[1]-np.count_nonzero(weights) 
#print("Number of weights with value zero".format(num_zero_w))

* From the above plot, it is also obvious that regularization is not required.

# Training with regularization 
* However, what happens to the performance of the model if we penalize, out of temptation, the weight values even to a smaller degree.
* Think about it.



In [ ]:
bin_sgd_clf_l2 = SGDClassifier(loss='log',
                               penalty='l2',
                               eta0=0.01,
                               alpha=0.001,
                               max_iter=1,
                               warm_start=True,
                               learning_rate='constant',
                               random_state=1729
                               )

Loss =[] 
iterations =100
for i in range(iterations):
  bin_sgd_clf_l2.fit(x_train, y_train_0)
  y_pred = bin_sgd_clf_l2.predict_proba(x_train)
  Loss.append(log_loss(y_train_0,y_pred))

In [ ]:
plt.figure() 
plt.plot(np.arange(iterations),Loss)
plt.grid(True) 
plt.xlabel('Iterations') 
plt.ylabel('Loss') 
plt.show()

In [ ]:
#calculation of weights and bias
weights = bin_sgd_clf_l2.coef_ 
bias = bin_sgd_clf_l2.intercept_

print(weights.shape, bias)

In [ ]:
plt.figure() 
plt.plot(np.arange(0,784),weights[0,:])
plt.ylim(np.min(weights[0]-3),np.max(weights[0])+3)
plt.xlabel('Feature Index')
plt.ylabel('Weight Value')
plt.grid(True) 
plt.show()

In [ ]:
# zero weights calculation Note: zero weights can't contribute to features.
num_zero_w = len(np.where(weights==0)[0])
print('Number of zero weight count:',num_zero_w)


In [ ]:
# Training and testing accuracy 
print('Training accuracy: {0:.2f}'.format(bin_sgd_clf_l2.score(x_train,y_train_0)))
print('Testing accuracy: {0:.2f}'.format(bin_sgd_clf_l2.score(x_test,y_test_0)))

In [ ]:
y_hat_train_0 = bin_sgd_clf_l2.predict(x_train)
cm_display = ConfusionMatrixDisplay.from_predictions(y_train_0,y_hat_train_0,values_format='.5g')

In [ ]:
print(classification_report(y_train_0,y_hat_train_0))

#Displaying input image and its prediction 


In [ ]:
index = 7 # try some other index
plt.imshow(x_test[index,:].reshape(28,28),cmap='plasma')
plt.colorbar() 
pred = bin_sgd_clf_l2.predict(x_test[index].reshape(1,-1))
plt.title(str(pred))
plt.show()


Let's plot a few images and their respective **predictions** with SGDClassifier without regularization.


In [ ]:
y_hat_test_0 = bin_sgd_clf.predict(x_test) 
num_images = 9 # choose a square number 
factor = np.int(np.sqrt(num_images)) 
fig,ax = plt.subplots(nrows=factor, ncols = factor, figsize=(8,6))
idx_offset  = 0 # display "num_images" starting from idx_offset
for i in range(factor):
  index = idx_offset + i*(factor)
  for j in range(factor):
    ax[i,j].imshow(x_test[index+j].reshape(28,28),cmap='plasma')# we should not use x_train_with_
    ax[i,j].set_title("Prediction:{0}".format(str(y_hat_test_0[index+j])))
    ax[i,j].set_axis_off()

In [ ]:
indx_0 = np.where(y_test_0==1) 


In [ ]:
zeroImgs= x_test[indx_0[0]]
zeroLabls = y_hat_test_0[indx_0[0]]
num_images = 9 # choose a square number 
factor = np.int(np.sqrt(num_images)) 
fig,ax = plt.subplots(nrows=factor, ncols = factor, figsize=(8,6))
idx_offset  = 0 # display "num_images" starting from idx_offset
for i in range(factor):
  index = idx_offset + i*(factor)
  for j in range(factor):
    ax[i,j].imshow(zeroImgs[index+j].reshape(28,28),cmap='plasma')# we should not use x_train_with_
    ax[i,j].set_title("Prediction:{0}".format(str(zeroLabls[index+j])))
    ax[i,j].set_axis_off()

#Hyper-parameter tuning
* We have to use cross-validate folds and mesure the same metrics across these folds for different values of hyper-parameters.
* Logistic regression uses **sgd** solver and hence the learning rate and regularization rate are two important hyper-parameters.
* For the moment, we skip penalizing the parameters of the model and just search for a better learning rate using `RandomizedSearchCV() and draw the value from the uniform distribution.


In [ ]:
lr_grid = loguniform(1e-2,1e-1)


* Note that, `lr_grid` is an object that contains a method called `rvs()` which can be used to get the samples of given size.
* Therefore, we pass this `lr_grid` object to `RandomizedSearchCV()`. Internally, it makes use of this `rvs()` method for sampling.


In [ ]:
print(lr_grid.rvs(3,random_state=42))

In [ ]:
#estimator for convenience 
estimator=SGDClassifier(loss='log',
                        penalty='l2',
                        max_iter=1,
                        warm_start=True, 
                        eta0=0.01,
                        alpha=0,
                        learning_rate='constant',
                        random_state=1729)


In [ ]:
scores = RandomizedSearchCV(estimator,
                            param_distributions={'eta0':lr_grid},
                            cv=5,
                            scoring=['precision','recall','f1'],
                            n_iter=5,
                            refit='f1')


In [ ]:
#It take quite a long time to finish
scores.fit(x_train,y_train_0)

In [ ]:
pprint(scores.cv_results_)

* Let us pick the best estimator from the results

In [ ]:
best_bin_clf = scores.best_estimator_


In [ ]:
y_hat_train_best_0 = best_bin_clf.predict(x_train)

In [ ]:
print(classification_report(y_train_0, y_hat_train_best_0))

#Classification Report
**Precision/Recall Tradeoff** 


In [ ]:
y_scores = bin_sgd_clf.decision_function(x_train)
precisions, recalls, thresholds = precision_recall_curve(y_train_0,y_scores)
plt.figure(figsize=(10,4)) 
plt.plot(thresholds,precisions[:-1],'r--',label='precisions')
plt.plot(thresholds,recalls[:-1],'b-',label='recalls')
plt.legend(loc='upper right')
plt.grid(True)
plt.xlabel('thresholds')
plt.show()

In [ ]:
#precision recall curve 
plt.figure(figsize=(10,4))
plt.plot(recalls[:-1],precisions[:-1], 'b-')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

#The ROC Curve 


In [ ]:
fpr,tpr,thresholds = roc_curve(y_train_0,y_scores)
plt.figure(figsize=(10,4))
plt.plot(fpr,tpr,linewidth=2,label='Perceptron')
plt.plot([0,1],[0,1],'k--',label='Best_estimator')
plt.xlabel("False Positive rate")
plt.ylabel("True Positive rate")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
#roc auc curve 

auc=roc_auc_score(y_train_0,y_scores)
print('AUC: {0:.3f}'.format(auc))

#Logistic Regression 
* In the previous setup, we used `SGDClassifier` to train 0-detector model in an iterative manner.
 * We can also train such a classifier by solving a set of equations obtained by setting the derivative of loss w.r.t. weights to 0.
 * These are not linear equations and therefore we need a different set of solvers.

* Sklearn uses solvers like `liblinear`, `newton-cg`, `sag` `saga` and `lbfgs` to find the optimal weights.
* Regularization is applied by default.
* Parameters: 
  `LogisticRegression(penalty='l2',*,dual=False, tol=0.0001, c=1.0, fit_intercept = True, intercept_scaling=1, class_weight=None, solver='lbfgs',max_iter=100,multi_class='auto', verbose=0, warm_start=False, n_jobs=None, l1_ratio =None)` 
* Note some of the important default parameters:
  * Regularization: `penalty='l2'`
  * Regularization rate: `C=1`
  * Solver: `solver = 'lbfgs'`
* Let's implement LogisticRegression(), **without regularization** by setting the parameter $ C= \infty $. Therefore, we may expect performance close to `SGDClassifier` without regularization.

# Training without regularization 
* **STEP 1:** Instantiate a pipeline object with two stages:
 * The first stage contains `MinMaxScaler` for scaling the input.
 * The second state contains a `LogisticRegression` classifier with the regularization rate $C = \infty $
* **STEP 2:** Train the pipeline with feature matrix `x_train` and label vector `y_train_0`.



In [ ]:
pipe_logit = make_pipeline(MinMaxScaler(), LogisticRegression(random_state=1729,
                                                              solver='lbfgs',
                                                              C=np.infty))
pipe_logit.fit(x_train, y_train_0)


By executing this cell, we trained our `LogisticRegression` classifier, which can be used for making predictions on the new inputs.


#Hyperparameter search 

with GridSearchCV 


In the previous cell we trained `LogisticRegression` Classifier with default parameterization.

Now we will demonstrate how to search for the best parameter value for regularization rate C, as an illustration, using GridSearachCV.

   Note that you can also use `RandomizedSearchCV` for this purpose.

In order to use `GridSearchCV`, we first define a set of values that we want to try out for c. The best value of c will be found from this set.

We define the `pipeline` object exactly like how we defined in the previous cell while using `LogisticRegression` classifier with default parameters and no regularization.

The additional step here is to instantiate a `GridSearchCV` object with a `pipeline` estimator, parameter grid specification and f1 as a scoring function.

  Note that you can use other scoring functions like `precision`, `recall`, however the value of C is found such that the given scoring function is optimized.

  







In [ ]:
from sklearn.pipeline import Pipeline 

grid_Cs = [0,1e-4,1e-3,1e-2,1e-1,1e0,1e1,1e2]


scaler = MinMaxScaler() 
logreg = LogisticRegression(C=1.0, random_state =1729)

pipe = Pipeline([('scaler',scaler),
                 ('logistic',logreg)])

pipe_logit_cv = GridSearchCV(pipe, 
                             param_grid={'logistic__C':grid_Cs},
                             scoring='f1')
pipe_logit_cv.fit(x_train,y_train_0)

The `GridSearchCV` finds the best value of c and refits the estimator by default on the entire training set. This gives us the logistic regression classifier with best value of C.

We can check the value of the bst parameter by accessing the `best_params_` member variable of the `GridSearchCV` object.


In [ ]:
pipe_logit_cv.best_params_

and the best score can be found in `best_score_` member variable and can be obtained as follows:



In [ ]:
pipe_logit_cv.best_score_

The best estimator can be accessed with `best_estimator_` member variable.


In [ ]:
pipe_logit_cv.best_estimator_

#With `LogisticRegressionCV` 

Instead of using `GridSearchCV` for finding the best value for parameter c, we can use `LogisticRegressionCV` for performing the same job. 
  * **STEP 1**: Here we make use of `LogisticRegressionCV` estimator with number of cross validation folds `cv=5` and scoring scheme `scoring='f1'` in the `pipeline` object.

  * **STEP 2**: In the second step, we train the pipeline object as before.
  

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
estimator = LogisticRegressionCV(cv=5, scoring='f1',random_state=1729)
logit_cv = make_pipeline(MinMaxScaler(),estimator)
logit_cv.fit(x_train,y_train_0)

By default, `LogisticRegressionCV` refits the model on the entire training set with the best parameter values obtained via cross validation.

#Performance evaluation.

Precision, recall, f1-score 

Let's evaluate performance of these three different logistic regression classifiers for detecting digit 0 from the image.

 * Logistic regression without regularization 
 * Best logistic regression classifier found through `GridSearchCV`.
 * Best classifier found through `LogisticRegressionCV`.
  Note that `GridSearchCV` and `LogisticRegressionCV` by default refit the classifier for the best hyperparameter values. 

Let's get prediction for test set with these three classifiers: 



In [ ]:
lr_y_hat_0 = pipe_logit.predict(x_test) 
lr_gs_y_hat_0 = pipe_logit_cv.best_estimator_.predict(x_test)
lr_cv_y_hat_0 = logit_cv.predict(x_test) 

We will compare **Precision, recall and F1 score** for the three classifiers.

In [ ]:
precision_lr = precision_score(y_test_0, lr_y_hat_0) 
recall_lr = recall_score(y_test_0, lr_y_hat_0) 

precision_lr_gs = precision_score(y_test_0,lr_gs_y_hat_0)
recall_lr_gs = recall_score(y_test_0, lr_gs_y_hat_0)

precision_lr_cv = precision_score(y_test_0, lr_cv_y_hat_0)
recall_lr_cv = recall_score(y_test_0, lr_cv_y_hat_0)

In [ ]:
print(f"LogReg: precision={precision_lr},recall={recall_lr}")
print(f"GridSearch: precision={precision_lr_gs},recall={recall_lr_gs}")
print(f"LogRegCV: precision={precision_lr_cv},recall={recall_lr_cv}")

Note that all three classifiers have roughly the same performance as measured with precision and recall.
 * The `LogisticRegression` classifier obtained through `GridSearchCV` has the highest precision-marginally higher that the other two classifiers.
 * The `LogisticRegression` classifier obtained through `LogisticRegressionCV` has the highest recall - marginally higher than the other two classifiers.

Using PR-curve


In [ ]:
y_scores_lr = pipe_logit.decision_function(x_test)
precisions_lr, recalls_lr, thresholds_lr= precision_recall_curve(y_test_0, y_scores_lr)

y_scores_lr_gs = pipe_logit_cv.decision_function(x_test)
precisions_lr_gs, recalls_lr_gs, thresholds_lr_gs= precision_recall_curve(y_test_0, y_scores_lr_gs)

y_scores_lr_cv = pipe_logit_cv.decision_function(x_test)
precisions_lr_cv, recalls_lr_cv, thresholds_lr_cv= precision_recall_curve(y_test_0, y_scores_lr_cv)





We have all the quantities for plotting the PR curve. Let's plot PR curve for all three classifiers.

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(recalls_lr, precisions_lr, 'r--',label='LogReg')
plt.plot(recalls_lr_gs, precisions_lr_gs,'b-',label="GridSearchCV") 
plt.plot(recalls_lr_cv, precisions_lr_cv,'k--',label='LogRegCV')

plt.grid(True)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend() 
plt.show()


Note that the PR curves for all three classifiers overlap significantly.

Let's calculate are under the PR curve:


In [ ]:
from sklearn.metrics import auc 
auc_lr =auc(recalls_lr, precisions_lr)
auc_lr_gs=auc(recalls_lr_gs, precisions_lr_gs)
auc_lr_cv =auc(recalls_lr_cv, precisions_lr_cv)


In [ ]:
print('AUC-PR for logistic regression:',auc_lr)
print("AUC-PR for GridSearchCV",auc_lr_gs) 
print("AUC-PR for Logistic Regression CV:",auc_lr_cv)

Observe that the AUC for all three classifier is roughly the same with `LogisticRegression` classifier obtained through cross validation and grid search have slightly higher AUC under PR curve.

#Confusion Matrix 
We show a confusion matrix for test set with logistic regression classifier:


In [ ]:
cm_display = ConfusionMatrixDisplay.from_predictions(y_test_0,lr_y_hat_0, values_format='.5g')
plt.show()

#Confusion matrix for test set with logistic regression classifier obtained through **Grid search**: 



In [ ]:
cm_display = ConfusionMatrixDisplay.from_predictions(y_test_0, lr_gs_y_hat_0)

Confusion matrix for test set with logistic regression classifier through cross validation:


In [ ]:
cm_display =ConfusionMatrixDisplay.from_predictions(y_test_0,lr_cv_y_hat_0)

**Excercise** Plot ROC curve for all three classifiers and calculate area under ROC curve.

#Ridge Classifier 
* Ridge classifier cast the problem as least-square classification and finds the optimal weight using some matrix decomposition technique such as SVD.
* To train the ridge classifier, the labels should be $ y \in \{+1,-1\}$
* The classifier also by default implements L2 regularization. However, we first implement it without regularization by setting `alpha=0`

In [ ]:
#initialize new variable names with all -1
y_train_0 = -1*np.ones((len(y_train)))
y_test_0 = -1*np.ones((len(y_test)))

#find indices of digit 0 image
indx_0 = np.where(y_train=='0')
y_train_0[indx_0]=1
indx_0 = np.where(y_test=='0')
y_test_0[indx_0]=1


* First take a look into the parameters of the class
  `RidgeClassifier(alpha=1.0,*,fit_intercept=True, normalize='deprecated',copy_X=True,max_iter=None, tol=0.001,class_weight=None, solver='auto', positive=False,random_state=None)` 

* Note the parameter `normalize` is deprecated. 


In [ ]:
estimator = RidgeClassifier(normalize=False,alpha=0)
pipe_ridge=make_pipeline(MinMaxScaler(),estimator)
pipe_ridge.fit(x_train,y_train_0)

In [ ]:
#Performance 

y_hat_test_0 = pipe_ridge.predict(x_test) 
print(classification_report(y_test_0,y_hat_test_0))

#Cross Validation 


In [ ]:
cv_bin_ridge_clf = cross_validate(pipe_ridge,x_train,y_train_0, cv =5, 
                                  scoring=['precision','recall','f1'],
                                  return_train_score=True,
                                  return_estimator=True)
pprint(cv_bin_ridge_clf)


In [ ]:
best_estimator_id = np.argmax(cv_bin_ridge_clf['train_f1']); best_estimator_id

In [ ]:
best_estimator = cv_bin_ridge_clf['estimator'][best_estimator_id]

Let's evaluate the performance of the best clasifier on the test set:


In [ ]:
y_hat_test_0 = best_estimator.predict(x_test)
print(classification_report(y_test_0,y_hat_test_0))

#Further exploration
Let's see what these classifiers learnt about digit 0.

In [ ]:
models = (bin_sgd_clf,bin_sgd_clf_l2,pipe_logit,pipe_ridge)
titles =('sgd','regularized sgd','logit','ridge')
plt.figure(figsize=(4,4))
plt.subplots(2,2)
for i in range(0,4):
  if i<2:
    w = models[i].coef_
  else: 
    w = models[i][1].coef_ 
  w_matrix = w.reshape(28,28) 
  w_matrix[w_matrix < 0 ]=0 # just set the value less than zero to zero
  plt.subplot(2,2,i+1)
  plt.imshow(w_matrix,cmap='plasma') 
  plt.title(titles[i])
  plt.axis('off')
  plt.grid(False) 
  plt.colorbar()
  
plt.show()

#Multiclass Classifier (One vs ALL)

# Multiclass Logit with SGD 


In [ ]:
estimator = SGDClassifier(loss='log',
                          penalty='l2',
                          max_iter=1,
                          warm_start=True,
                          eta0=0.01,
                          alpha=0,
                          learning_rate='constant',
                          random_state=1729) 
pipe_sgd_ovr = make_pipeline(MinMaxScaler(),estimator) 


In [ ]:
Loss = [] 
iterations = 100 
for i in range(iterations):
  pipe_sgd_ovr.fit(x_train,y_train) 
  y_pred = pipe_sgd_ovr.predict_proba(x_train) 
  Loss.append(log_loss(y_train,y_pred)) 

In [ ]:
plt.figure(figsize=(10,6)) 
plt.plot(np.arange(iterations),Loss) 
plt.grid(True) 
plt.xlabel('Iterations') 
plt.ylabel('Loss')
plt.show()

What actually happened behind the screen is that the library automatically created 10 binary classifiers and trained them ! During the interference time, the input will be passed through all the 10 classifiers and the highest score among the outputs will be considered as the predicted class. To see it in acction, let us execute the following lines of code.



In [ ]:
pipe_sgd_ovr[1]

In [ ]:
pipe_sgd_ovr[1].coef_.shape

So, it is a matrix of size $ 10 \times 784$. A row represents the weights of a single binary classifier.


In [ ]:
y_hat = pipe_sgd_ovr.predict(x_test) ; y_hat[:5]

In [ ]:
cm_display = ConfusionMatrixDisplay.from_predictions( y_test, y_hat, values_format='.5g' )

In [ ]:
print(classification_report(y_test,y_hat))

Multi-class LogisticRegression using solvers 


In [ ]:
pipe_logit_ovr = make_pipeline(MinMaxScaler(),LogisticRegression(random_state=1729,
                                                                 solver='lbfgs',
                                                                 C=np.infty)) 
pipe_logit_ovr.fit(x_train,y_train)

In [ ]:
y_hat = pipe_logit_ovr.predict(x_test) 
cm_display = ConfusionMatrixDisplay.from_predictions(y_test,y_hat, values_format='.5g')

In [ ]:
print(classification_report(y_test,y_hat))

#Visualize the weight values


In [ ]:
w = pipe_logit_ovr[1].coef_
# normalize
w =MinMaxScaler().fit_transform(w) 
plt.subplots(3,3)
for i in range(9):
  #w_i[w_i<0]=0
  plt.subplot(3,3,i+1)
  plt.imshow(w[i+1].reshape(28,28), cmap='plasma')
  plt.title('W_{0}'.format(i+1))
  plt.grid(False)
  plt.axis('off')
  plt.colorbar() 

plt.show()


# **Excercise** Multiclass classification with RidgeClassifier

In [ ]:
print(classification_report(y_test,y_hat))

# Text Classification with Naive Bayes classifier 
In this colab, we will use Naive Bayes classifier for classifying text.

Naive Bayes classifier is used for text classification and spam detection tasks.

Here is an example as how to perform the text classification with Naive Bayes classifier.

In [ ]:
#Data loading 
from sklearn.datasets import fetch_20newsgroups 
 
#Preprocessing 
from sklearn.feature_extraction.text import TfidfVectorizer

#Model/estimator 
from sklearn.naive_bayes import MultinomialNB 

#Pipeline utility 
from sklearn.pipeline import make_pipeline 

#Model evaluation 
from sklearn.metrics import ConfusionMatrixDisplay 

#plotting library 
import matplotlib.pyplot as plt


Exercise read about **TfidfVectorizer** API

#Dataset 

We will be using 20 newsgroup data set for classification 

As a first step, let's download 20 newsgroup dataset with `fetch_20newsgroup`API.



In [ ]:
data = fetch_20newsgroups() 


Let's look at the names of the classes 


In [ ]:
data.target_names

There are **20 categories** in the dataset. For simplicity, we will select **4** of these categories and download training and test sets.


In [ ]:
categories = ['talk.religion.misc','soc.religion.christian','sci.space','comp.graphics'] 
train = fetch_20newsgroups(subset='train', categories=categories) 
test = fetch_20newsgroups(subset='test',categories=categories) 

Let's look at a sample trianing document: 


In [ ]:
print(train.data[5])

This data is different than what we have seen so far. Here the training data contains document in text form.
# Data preprocessing and modeling 

As we have mentioned this in the first week of machine learning techniques course, we need to convert the text data to numeric form.
`TfidfVectorizer` is one such API that converts text input into a vector of numerical values.

We will use `TfidfVectorizer` as a preprocessing step to obtain feature vector corresponding to the text document.

We will be using multinomial naive Bayes Classifier for categorizing documents from 20newsgroup corpus. 




In [ ]:
model = make_pipeline(TfidfVectorizer(),MultinomialNB()) 

In [ ]:
model.fit(train.data,train.target)

#Model evaluation 

Let's first predict the labels for the test set and then calculate the confusion matrix for the test set.


In [ ]:
ConfusionMatrixDisplay.from_estimator(model,test.data, test.target, display_labels=test.target_names,xticks_rotation=30) 
plt.show()

Observe that: 
  * There is a confusion between documents of class `soc.religion.christian` and `talk.religion.misc`, which is along the expected lines.
  * The classes `comp.graphics` and `sci.space` are well separated by such a simple classifier.

Now we have a tool to classify statements into one of these four classes.
  Make use of `predict` function on pipeline for predicting category of a test string.
   

In [ ]:
def predict_category(s, train = train, model=model):
  pred = model.predict([s]) 
  return train.target_names[pred[0]]

Using this function for prediction:

In [ ]:
predict_category('sending a payload to this ISS') 


In [ ]:
predict_category('discussing islam vs atheism')

In [ ]:
predict_category('determining the screen resolution')